### 每月销售额总量预测

### Connect postgresql database

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text

# 数据库配置
username = "XXXXXX"
password = "YYYYYY"
host = "localhost"
port = 5432
database = "eyewear-data"

# 创建连接
engine = create_engine(
    f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}"
)


### Query and cal customer RFM feature

In [ ]:
# 1 . 每月销售额、订单数量和平均订单价值
sale_amount_monthly_sql = """
WITH cluster_region_monthly_quantity AS (
    SELECT
        TO_CHAR(o.order_date,'YYYY-MM') AS order_month,
        c.customer_hierarchy,
        c.customer_type,
        c.region,
        SUM(CASE WHEN oi.line_price_after_tax > 0 THEN oi.quantity ELSE 0 END) AS region_paid_quantity
    FROM "Order" o
    JOIN "OrderItem" oi ON o.order_id = oi.order_id
    LEFT JOIN "CustomerInfo" c ON o.customer_id = c.customer_id
    WHERE o.order_status IN ('Completed','Shipped')
      AND o.order_date IS NOT NULL 
      AND c.customer_hierarchy IS NOT NULL 
    GROUP BY order_month, c.customer_hierarchy, c.customer_type, c.region
),

-- 地区集中度（基于销量）
cluster_region_rank AS (
    SELECT *,
           RANK() OVER(
           PARTITION BY order_month,customer_hierarchy, customer_type ORDER BY region_paid_quantity DESC
           ) AS region_rank
    FROM cluster_region_monthly_quantity   -- ← 这里修正：改成 cluster_region_monthly_quantity
),

cluster_region_concentration AS (
    SELECT
        order_month,customer_hierarchy, customer_type,
        MAX(CASE WHEN region_rank = 1 THEN region_paid_quantity END)::numeric 
            / NULLIF(SUM(region_paid_quantity)::numeric, 0) AS top_region_quantity_ratio,
        SUM(CASE WHEN region_rank <=5 THEN region_paid_quantity ELSE 0 END)::numeric 
            / NULLIF(SUM(region_paid_quantity)::numeric, 0) AS top5_region_quantity_ratio
    FROM cluster_region_rank
    GROUP BY order_month, customer_hierarchy, customer_type
),

cluster_region_entropy AS (
    SELECT
        order_month,
        customer_hierarchy,
        customer_type,
        -SUM(qty_share * LN(qty_share)) AS region_paid_qty_entropy
    FROM (
        SELECT
            order_month,
            customer_hierarchy, 
            customer_type,
            region,
            region_paid_quantity::numeric / NULLIF(SUM(region_paid_quantity::numeric) 
            OVER (PARTITION BY order_month, customer_hierarchy, customer_type), 0) AS qty_share
        FROM cluster_region_monthly_quantity   -- ← 这里也要修正
    ) t
    GROUP BY order_month, customer_hierarchy, customer_type
)

SELECT
    to_char(o.order_date, 'YYYY-MM') AS order_month,
    -- 基础指标
    c.customer_hierarchy,
    c.customer_type,

    COUNT(DISTINCT o.order_id) AS order_count,
    COUNT(DISTINCT o.customer_id) AS unique_customer_count,

    SUM(oi.line_price_after_tax) AS monthly_revenue,
    SUM(oi.line_price_before_tax) AS monthly_revenue_before_tax,
    SUM(o.total_price_after_tax)/COUNT(DISTINCT o.order_id) AS avg_order_value,

    -- 目标变量
    SUM(oi.quantity) AS total_quantity,   
    SUM(CASE WHEN oi.line_price_after_tax > 0 THEN oi.quantity ELSE 0 END) AS customer_paid_quantity, -- ← 预测目标
    
    -- 每个订单平均付费商品数量（推荐）
    SUM(CASE WHEN oi.line_price_after_tax > 0 THEN oi.quantity ELSE 0 END)::numeric 
        / NULLIF(COUNT(DISTINCT o.order_id), 0) AS avg_paid_qty_per_order,

    -- 每个订单平均总商品数量（包含赠品）
    SUM(oi.quantity)::numeric 
        / NULLIF(COUNT(DISTINCT o.order_id), 0) AS avg_total_qty_per_order,

    -- 价格
    SUM(oi.line_price_after_tax)/NULLIF(SUM(CASE WHEN oi.line_price_after_tax > 0 THEN oi.quantity ELSE 0 END),0) 
        AS avg_unit_price,

    -- 促销
    COUNT(DISTINCT pa.campaign_id) AS promotion_count,
    COUNT(DISTINCT CASE WHEN o.campaign_id IS NOT NULL THEN o.order_id END) AS promo_order_count,
    -- 促销订单占比（按订单）
    COUNT(DISTINCT CASE WHEN o.campaign_id IS NOT NULL THEN o.order_id END)::numeric 
        / NULLIF(COUNT(DISTINCT o.order_id), 0) AS promo_order_ratio,
    
    -- 促销贡献的付费销量占比（推荐使用这个）
    SUM(CASE WHEN o.campaign_id IS NOT NULL 
             THEN (CASE WHEN oi.line_price_after_tax > 0 THEN oi.quantity ELSE 0 END)
             ELSE 0 END)::numeric 
        / NULLIF(SUM(CASE WHEN oi.line_price_after_tax > 0 THEN oi.quantity ELSE 0 END), 0) 
        AS promo_paid_qty_ratio,

    -- 促销贡献的总销量占比（作为对比）
    SUM(CASE WHEN o.campaign_id IS NOT NULL THEN oi.quantity ELSE 0 END)::numeric 
        / NULLIF(SUM(oi.quantity), 0) AS promo_total_qty_ratio,
      
    AVG(CASE WHEN pa.discount_type='Percentage' THEN pa.discount_value END) AS avg_percentage_discount,

    COUNT(DISTINCT CASE WHEN pa.discount_type = 'Percentage' THEN o.order_id END)::numeric 
        / NULLIF(COUNT(DISTINCT o.order_id), 0) AS percentage_discount_order_ratio,

    COUNT(DISTINCT CASE WHEN pa.discount_type = 'Fixed Amount' THEN o.order_id END)::numeric 
        / NULLIF(COUNT(DISTINCT o.order_id), 0) AS fixed_amount_discount_order_ratio,

    COUNT(DISTINCT CASE WHEN pa.discount_type = 'Free Gift' THEN o.order_id END)::numeric 
        / NULLIF(COUNT(DISTINCT o.order_id), 0) AS free_gift_order_ratio,

    COUNT(DISTINCT CASE WHEN pa.discount_type = 'Buy One Get One' THEN o.order_id END)::numeric 
        / NULLIF(COUNT(DISTINCT o.order_id), 0) AS bogo_order_ratio,

    -- 地区
    MAX(rc.top_region_quantity_ratio) AS top_region_quantity_ratio,
    MAX(rc.top5_region_quantity_ratio) AS top5_region_quantity_ratio,
    MAX(re.region_paid_qty_entropy) AS region_paid_qty_entropy,

    -- 产品结构（已使用quantity）
    SUM(CASE WHEN p.category = 'Eyeglasses' 
            THEN (CASE WHEN oi.line_price_after_tax > 0 THEN oi.quantity ELSE 0 END) 
            ELSE 0 END) AS paid_qty_eyeglasses,

    SUM(CASE WHEN p.category = 'Sunglasses' 
            THEN (CASE WHEN oi.line_price_after_tax > 0 THEN oi.quantity ELSE 0 END) 
            ELSE 0 END) AS paid_qty_sunglass,

    SUM(CASE WHEN p.category = 'AI Glasses' 
            THEN (CASE WHEN oi.line_price_after_tax > 0 THEN oi.quantity ELSE 0 END) 
            ELSE 0 END) AS paid_qty_ai_glasses,

    SUM(CASE WHEN p.category = 'Lens' 
            THEN (CASE WHEN oi.line_price_after_tax > 0 THEN oi.quantity ELSE 0 END) 
            ELSE 0 END) AS paid_qty_lens,

    -- 时间特征
    EXTRACT(YEAR FROM o.order_date) AS year,
    EXTRACT(MONTH FROM o.order_date) AS month,
    CONCAT(EXTRACT(YEAR FROM o.order_date),'-Q',EXTRACT(QUARTER FROM o.order_date)::int) AS quarter,
    CASE 
        WHEN EXTRACT(MONTH FROM o.order_date) IN (12,1,2) THEN 'Winter'
        WHEN EXTRACT(MONTH FROM o.order_date) IN (3,4,5) THEN 'Spring'
        WHEN EXTRACT(MONTH FROM o.order_date) IN (6,7,8) THEN 'Summer'
        ELSE 'Autumn' 
    END AS season

FROM "Order" o
JOIN "OrderItem" oi ON o.order_id = oi.order_id
LEFT JOIN "PromotionActivity" pa ON o.campaign_id = pa.campaign_id
LEFT JOIN "CustomerInfo" c ON o.customer_id = c.customer_id
LEFT JOIN "ProductInfo" p ON oi.product_id = p.product_id
LEFT JOIN cluster_region_concentration rc 
ON rc.order_month = TO_CHAR(o.order_date,'YYYY-MM') 
AND rc.customer_hierarchy = c.customer_hierarchy 
AND rc.customer_type = c.customer_type
LEFT JOIN cluster_region_entropy re 
ON re.order_month = TO_CHAR(o.order_date,'YYYY-MM')
AND re.customer_hierarchy = c.customer_hierarchy 
AND re.customer_type = c.customer_type
WHERE o.order_status IN ('Completed', 'Shipped')
  AND o.order_date IS NOT NULL
  AND p.category <> 'Accessory'

GROUP BY
  to_char(o.order_date, 'YYYY-MM'),
  c.customer_hierarchy,
  c.customer_type,
  EXTRACT(YEAR FROM o.order_date),
  EXTRACT(MONTH FROM o.order_date),
  EXTRACT(QUARTER FROM o.order_date),
  CASE 
    WHEN EXTRACT(MONTH FROM o.order_date) IN (12,1,2) THEN 'Winter'
    WHEN EXTRACT(MONTH FROM o.order_date) IN (3,4,5) THEN 'Spring'
    WHEN EXTRACT(MONTH FROM o.order_date) IN (6,7,8) THEN 'Summer'
    ELSE 'Autumn' 
  END
"""

df_sale_amount_monthly = pd.read_sql(sale_amount_monthly_sql, engine)

df_sale_amount_monthly

In [ ]:
df_sale_amount_monthly['customer_hierarchy'].unique()

In [ ]:
df_sale_amount_monthly.columns

### cal timeseries feature for sale amount(针对热销产品预测销售金额)

In [ ]:
import numpy as np

df_timeseries_features = (
    df_sale_amount_monthly[
        ["order_month","customer_hierarchy","customer_type", "monthly_revenue","order_count","unique_customer_count"]
    ]
    .sort_values(["order_month","customer_hierarchy","customer_type"])
    .reset_index(drop=True)
    .copy()
)

df_timeseries_features["trend"] = (
    df_timeseries_features
    .groupby(["customer_hierarchy","customer_type"])
    .cumcount()
)


# 基础时间序列特征
for lag in [1, 2, 3, 4, 12]:
    df_timeseries_features[f"lag_{lag}_month_revenue"] = (
        df_timeseries_features
        .groupby(["customer_hierarchy","customer_type"])["monthly_revenue"]
        .shift(lag)
    )

# 滚动平均
def rolling_mean(s, window):
    return s.shift(1).rolling(window, min_periods=1).mean()

df_timeseries_features["rolling_3_month_avg_revenue"] = (
    df_timeseries_features
    .groupby(["customer_hierarchy", "customer_type"])["monthly_revenue"]
    .transform(lambda x: rolling_mean(x, 3))
)

df_timeseries_features["rolling_6_month_avg_revenue"] = (
    df_timeseries_features
    .groupby(["customer_hierarchy", "customer_type"])["monthly_revenue"]
    .transform(lambda x: rolling_mean(x, 6))
)

df_timeseries_features["rolling_12_month_avg_revenue"] = (
    df_timeseries_features
    .groupby(["customer_hierarchy", "customer_type"])["monthly_revenue"]
    .transform(lambda x: rolling_mean(x, 12))
)


# ==================== 新增的3种 Baseline ====================

# 1. Historical Mean（历史均值）
df_timeseries_features["historical_mean"] = (
    df_timeseries_features
    .groupby(["customer_hierarchy", "customer_type"])["monthly_revenue"]
    .transform(lambda x: x.expanding().mean())
)

# 2. Historical Median（历史中位数 Baseline）
df_timeseries_features["historical_median"] = (
    df_timeseries_features
    .groupby(["customer_hierarchy", "customer_type"])["monthly_revenue"]
    .transform(lambda x: x.expanding().median())
)

# 3. Weighted Moving Average (WMA - 线性加权移动平均)
def weighted_moving_average(x, window=3):
    """线性加权：最近的月份权重最高"""
    if len(x) < window:
        return np.nan
    weights = np.arange(1, window + 1)      # 如 window=3 时权重为 [1,2,3]
    return np.dot(x, weights) / weights.sum()

# 计算 WMA
df_timeseries_features["wma_3"] = (
    df_timeseries_features
    .groupby(["customer_hierarchy", "customer_type"])["monthly_revenue"]
    .transform(lambda x: x.shift(1).rolling(3).apply(weighted_moving_average, raw=True))
)

# 指数加权移动平均
df_timeseries_features["ewm_3"] = (
    df_timeseries_features
    .groupby(["customer_hierarchy","customer_type"])["monthly_revenue"]
    .transform(lambda x: x.ewm(span=3,adjust=False).mean())
)

df_timeseries_features["ewm_6"] = (
    df_timeseries_features
    .groupby(["customer_hierarchy","customer_type"])["monthly_revenue"]
    .transform(lambda x: x.ewm(span=6,adjust=False).mean())
)

df_timeseries_features["ewm_alpha"] = (
    df_timeseries_features
    .groupby(["customer_hierarchy","customer_type"])["monthly_revenue"]
    .transform(lambda x: x.ewm(alpha=0.3,adjust=False).mean())
)

# Growth 特征
df_timeseries_features["mom_growth"] = (
    df_timeseries_features
    .groupby(["customer_hierarchy", "customer_type"])["monthly_revenue"]
    .transform(lambda x: x.shift(1).pct_change(1))
)

df_timeseries_features["yoy_growth"] = (
    df_timeseries_features
    .groupby(["customer_hierarchy", "customer_type"])["monthly_revenue"]
    .transform(lambda x: x.shift(1).pct_change(12))
)

# 加速/减速信号
df_timeseries_features["mom_acceleration"] = (
    df_timeseries_features["mom_growth"]
    - df_timeseries_features
        .groupby(["customer_hierarchy", "customer_type"])["mom_growth"]
        .shift(1)
)

df_timeseries_features["lag_1_order_count"] = (
    df_timeseries_features
    .groupby(["customer_hierarchy", "customer_type"])["order_count"]
    .shift(1)
)

df_timeseries_features["lag_1_customer_count"] = (
    df_timeseries_features
    .groupby(["customer_hierarchy", "customer_type"])["unique_customer_count"]
    .shift(1)
)

df_timeseries_features

In [ ]:
df_timeseries_features.columns

### merge 2 dataframe

In [ ]:
df_sale_amount_monthly_final = (
    df_sale_amount_monthly
    .merge(
        df_timeseries_features[
            [
                "order_month",
                "customer_hierarchy",
                "customer_type",
                "trend",
                "lag_1_month_revenue",
                "lag_2_month_revenue",
                "lag_3_month_revenue",
                "lag_4_month_revenue",
                "lag_12_month_revenue",
                "rolling_3_month_avg_revenue",
                "rolling_6_month_avg_revenue",
                "rolling_12_month_avg_revenue",
                "historical_mean", 
                "historical_median", 
                "wma_3",
                "ewm_3", 
                "ewm_6", 
                "ewm_alpha",
                "mom_growth",
                "yoy_growth",
                "mom_acceleration", 
                "lag_1_order_count",
                "lag_1_customer_count"
            ]
        ],
        on=["order_month","customer_hierarchy","customer_type"],
        how="left"
    )
)

df_sale_amount_monthly_final

In [ ]:
df_sale_amount_monthly_final.columns

In [ ]:
# 预测的目标值
target_cols = ['monthly_revenue']

# 预测需要使用的特征列
feature_cols = [
    "customer_hierarchy", 
    "customer_type",
    'lag_1_month_revenue', 
    'lag_3_month_revenue',
    'rolling_3_month_avg_revenue', 
    'rolling_6_month_avg_revenue',
    'mom_growth', 
    
    # 季节与周期
    'month', 'quarter', 

    # 历史业务量
    'lag_1_order_count',
    'lag_1_customer_count',
]

df_sale_predict_dataset = df_sale_amount_monthly_final[['order_month'] + target_cols + feature_cols].copy()
df_sale_predict_dataset

### Data Validation

In [ ]:
# 基本检查
print(df_sale_predict_dataset.shape)
# 缺失值情况
print(df_sale_predict_dataset.isnull().sum())
print(df_sale_predict_dataset.dtypes)

# 相关性（快速看特征重要性）
print(df_sale_predict_dataset.columns.tolist())
revenue_cols = [col for col in df_sale_predict_dataset.columns if 'revenue' in col]
print("所有包含 'revenue' 的列：", revenue_cols)

###  Imputing Missing Values (for Linear Models / Neural Networks Models)

In [ ]:
df_filled = df_sale_predict_dataset.sort_values(
    ['customer_hierarchy','customer_type', 'order_month']
).copy()

lag_cols = [
    'lag_1_month_revenue',
    'lag_3_month_revenue',
    'lag_1_order_count',
]
rolling_cols = [
    'rolling_3_month_avg_revenue',
    'rolling_6_month_avg_revenue',
]
# growth_cols = ['mom_growth', 'yoy_growth', 'mom_acceleration']

growth_cols = ['mom_growth']
# 1) 时间序列特征：按产品时间前向填，再补 0
ts_cols = lag_cols + rolling_cols
df_filled[ts_cols] = df_filled.groupby(['customer_hierarchy','customer_type'])[ts_cols].ffill()
df_filled[ts_cols] = df_filled[ts_cols].fillna(0)

# 2) 增长率：未知视为 0
df_filled[growth_cols] = df_filled[growth_cols].fillna(0)

# 3) 促销折扣均值：无折扣视为 0
if 'avg_percentage_discount' in df_filled.columns:
    df_filled['avg_percentage_discount'] = (
        df_filled['avg_percentage_discount'].fillna(0)
    )

print(df_filled.isnull().sum())
df_final = df_filled.sort_values('order_month').reset_index(drop=True)
df_final

### Imputing Missing Values debug 1 (for LightGBM / CatBoost / XGBoost)

In [ ]:
df_filled = df_sale_predict_dataset.sort_values(
    ['customer_hierarchy','customer_type', 'order_month']
).copy()

# 增长率：未知 → 0（可选）
# growth_cols = ['mom_growth', 'yoy_growth', 'mom_acceleration']
growth_cols = ['mom_growth']
df_filled[growth_cols] = df_filled[growth_cols].fillna(0)

# 无百分比折扣 → 0
# df_filled['avg_percentage_discount'] = df_filled['avg_percentage_discount'].fillna(0)

# lag / rolling：保留 NaN，交给 LightGBM/CatBoost
# 不要 numeric_cols.fillna(0)

df_final = df_filled.sort_values(['order_month']).reset_index(drop=True)
df_final

In [ ]:
# 基本检查
print(df_final.shape)
# 缺失值情况
print(df_final.isnull().sum())

### check `df_final_coverted` data

In [ ]:
# ==================== 数据类型转换（在分割之前） ====================
df_final_converted = df_final.copy()

# 先检查原始数据
print("转换前的数据类型:")
print(df_final_converted[['quarter' ]].dtypes)
print("\n转换前的唯一值:")
print("Quarter 唯一值:", df_final_converted['quarter'].unique())
print("\n是否存在空值:")
print(df_final_converted[['quarter']].isnull().sum())

# 2. 转换 quarter: 从 '2023-Q1' 提取季度数字 1-4
if df_final_converted['quarter'].dtype == 'object':
    # 方法1: 使用 str[0] 参数（推荐）
    df_final_converted['quarter'] = df_final_converted['quarter'].str.extract(r'Q(\d)', expand=False).astype(int)
    print("Quarter 转换完成")
else:
    print("\nWarning: quarter 列不是 object 类型")

# 验证转换结果
print("\n转换后的数据类型:")
print(df_final_converted[['quarter']].dtypes)
print("\n转换后的样例:")
print(df_final_converted[['order_month', 'quarter']].head(10))
print("\n转换后的唯一值:")
print("Quarter:", sorted(df_final_converted['quarter'].unique()))

# 使用转换后的数据进行分割
df_final = df_final_converted


### split dataset

In [ ]:
# 24个月数据建议划分:
# 训练集: 前18个月 (2023-01 ~ 2024-06)
# 测试集: 后6个月 (2024-07 ~ 2024-12)
# 这样可以保留约25%数据用于测试,符合业务季节性验证

# 按月份划分：最后 6 个自然月做测试集
n_test_months = 6

months = sorted(df_final['order_month'].unique())
print("全部月份数:", len(months))
print("月份范围:", months[0], "~", months[-1])

if len(months) <= n_test_months:
    raise ValueError(f"月份总数 {len(months)} 不足以留出 {n_test_months} 个月测试集")

train_months = months[:-n_test_months]
test_months = months[-n_test_months:]

train = df_final[df_final['order_month'].isin(train_months)].copy()
test = df_final[df_final['order_month'].isin(test_months)].copy()

# 保持时间顺序（可选再按 product_id）
train = train.sort_values(['order_month', 'customer_hierarchy', 'customer_type']).reset_index(drop=True)
test = test.sort_values(['order_month',  'customer_hierarchy', 'customer_type']).reset_index(drop=True)

print("训练集月份:", train['order_month'].min(), "~", train['order_month'].max())
print("测试集月份:", test['order_month'].min(), "~", test['order_month'].max())
print("训练月份列表:", train_months)
print("测试月份列表:", test_months)
print(f"训练集大小: {len(train)}, 测试集大小: {len(test)}")
print("训练集 cluster 数:", 
      train[['customer_hierarchy', 'customer_type']].drop_duplicates().shape[0])
print("测试集 cluster 数:", 
      test[['customer_hierarchy', 'customer_type']].drop_duplicates().shape[0])


In [ ]:
train

In [ ]:
train.columns

In [ ]:
test

### Separation of features and objectives

In [ ]:
target_revenue = 'monthly_revenue'

# feature_cols 已在上面定义了

# 分离特征和目标
X_train = train[feature_cols].copy()
y_train_revenue = train[target_revenue].copy()

X_test = test[feature_cols].copy()
y_test_revenue = test[target_revenue].copy()


In [ ]:
X_train

In [ ]:
y_train_revenue

### Run LightGBM

In [ ]:
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error


lgm_model_rev = lgb.LGBMRegressor(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=5,
    num_leaves=47,
    min_child_samples=8,
    min_child_weight=0.001,
    reg_alpha=0.1,
    reg_lambda=0.5,
    subsample=0.85,
    colsample_bytree=0.85,
    random_state=42,
    verbose=-1
)

cat_features = ['product_id', 'category']

for col in cat_features:
    if col in X_train.columns:
        X_train[col] = X_train[col].astype('category')
        X_test[col] = X_test[col].astype('category')

# 保证测试集类别与训练集一致（避免未见类别）
if 'category' in X_train.columns:
    X_test['category'] = pd.Categorical(
        X_test['category'],
        categories=X_train['category'].cat.categories
    )

# 使用eval_set监控测试集表现(但不用于早停,仅用于观察)
lgm_model_rev.fit(
    X_train, y_train_revenue,
    eval_X=X_test, 
    eval_y=y_test_revenue,
    eval_metric='mape',
)

# 预测和评估
lgm_pred_rev = lgm_model_rev.predict(X_test)
lgm_pred_rev_final = lgm_pred_rev

lgm_mape = mean_absolute_percentage_error(y_test_revenue, lgm_pred_rev_final)
lgm_mae = mean_absolute_error(y_test_revenue, lgm_pred_rev_final)
lgm_bias = (lgm_pred_rev_final.sum() - y_test_revenue.sum()) / y_test_revenue.sum()

print(f"Bias: {lgm_bias:.2%}")
print(f"Revenue MAPE: {lgm_mape:.2%}")
print(f"Revenue MAE: ${lgm_mae:,.2f}")

# ========== 新增：Bias 校准 ==========
# 计算训练集上的系统偏差
lgm_train_pred = lgm_model_rev.predict(X_train)
lgm_train_bias = (lgm_train_pred - y_train_revenue).mean()
print(f"\n训练集平均偏差: ${lgm_train_bias:,.0f}")

# 校准测试集预测
lgm_pred_rev_calibrated = lgm_pred_rev_final - lgm_train_bias

# ====================================

print("lgm_pred_rev:",lgm_pred_rev_final)
print("y_test_revenue:",y_test_revenue.values)

print("lgm_pred_rev min:", lgm_pred_rev_final.min())
print("lgm_pred_rev max:", lgm_pred_rev_final.max())

print("y_test_revenue min:", y_test_revenue.min())
print("y_test_revenue max:", y_test_revenue.max())

### debug run lightgbm + optuna

In [ ]:
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
import optuna

# ---------- 1) 训练期内再切验证（按月，不是按行）----------
n_val_months = 3  # 例如 2024-04~06 做 val，更早做 train_inner

train_months_sorted = sorted(train['order_month'].unique())
if len(train_months_sorted) <= n_val_months + 3:
    raise ValueError("训练月份太少，无法再切验证集")

inner_train_months = train_months_sorted[:-n_val_months]
val_months = train_months_sorted[-n_val_months:]

tr_mask = train['order_month'].isin(inner_train_months)
va_mask = train['order_month'].isin(val_months)

X_tr = X_train.loc[tr_mask].copy()
y_tr = y_train_revenue.loc[tr_mask].copy()
X_va = X_train.loc[va_mask].copy()
y_va = y_train_revenue.loc[va_mask].copy()

# 类别特征（与主流程一致）
cat_features = [c for c in ['customer_hierarchy', 'customer_type', 'quarter', 'month'] if c in X_tr.columns]

def cast_categorical(df, cat_cols, ref_df=None):
    """把类别列转成 category；若给了 ref_df，则用其 categories 对齐"""
    out = df.copy()
    for col in cat_cols:
        if ref_df is None:
            out[col] = out[col].astype("category")
        else:
            out[col] = pd.Categorical(
                out[col],
                categories=ref_df[col].astype("category").cat.categories
            )
    return out

X_tr = cast_categorical(X_tr, cat_features)
X_va = cast_categorical(X_va, cat_features, ref_df=X_tr)

print("inner train:", inner_train_months[0], "~", inner_train_months[-1], "n=", len(X_tr))
print("valid months:", val_months, "n=", len(X_va))
print("cat_features:", cat_features)


def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "learning_rate": trial.suggest_float("learning_rate", 0.03, 0.12, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 5),
        "num_leaves": trial.suggest_int("num_leaves", 8, 31),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 20),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 2.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 3.0),
        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
        "min_child_weight": 1e-3,
    }
    
    model = lgb.LGBMRegressor(**params, random_state=42, verbose=-1)
    model.fit(
        X_tr, y_tr,
        eval_X=X_va,
        eval_y=y_va,
        eval_metric='mape',
        categorical_feature=cat_features,
        callbacks=[lgb.early_stopping(80, verbose=False)]
    )
    pred = model.predict(X_va)
    return mean_absolute_percentage_error(y_va, pred)

# ====================== 运行 Optuna ======================
print("开始 Optuna 调参...")
study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=50)   # 可根据时间调整 trials 数量

print("Best MAPE:", study.best_value)
print("Best params:", study.best_params)

# ====================== 使用最优参数训练最终模型 ======================
best_params = study.best_params.copy()

# 类别：全量 train + test
X_train_fit = cast_categorical(X_train.copy(), cat_features)
X_test_fit = cast_categorical(X_test.copy(), cat_features, ref_df=X_train_fit)

lgm_model_rev = lgb.LGBMRegressor(
    **best_params,
    random_state=42,
    verbose=-1,
)

# 预测和评估
# 最终训练
lgm_model_rev.fit(X_train_fit, y_train_revenue)

# ---------- 4) 测试集只评估一次 ----------
lgm_pred_rev_final = lgm_model_rev.predict(X_test_fit)

lgm_mape = mean_absolute_percentage_error(y_test_revenue, lgm_pred_rev_final)
lgm_mae = mean_absolute_error(y_test_revenue, lgm_pred_rev_final)
lgm_bias = (lgm_pred_rev_final.sum() - y_test_revenue.sum()) / y_test_revenue.sum()

print(f"TEST Bias: {lgm_bias:.2%}")
print(f"TEST MAPE: {lgm_mape:.2%}")
print(f"TEST MAE: {lgm_mae:,.2f}")
# 偏差校准
lgm_train_pred = lgm_model_rev.predict(X_train_fit)
lgm_train_bias = (lgm_train_pred - y_train_revenue).mean()
print(f"训练集平均偏差: {lgm_train_bias:,.0f}")

lgm_pred_revenue_calibrated = lgm_pred_rev_final - lgm_train_bias

print("\n=== 最终预测 vs 真实值 ===")
print("预测值:", np.round(lgm_pred_revenue_calibrated, 2))
print("真实值:", y_test_revenue.values)
print("预测范围:", lgm_pred_revenue_calibrated.min(), "→", lgm_pred_revenue_calibrated.max())

### debug run lightgbm + optuna + log 

In [ ]:
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
import optuna

# ---------- 1) 训练期内再切验证（按月，不是按行）----------
n_val_months = 3  # 例如 2024-04~06 做 val，更早做 train_inner

train_months_sorted = sorted(train['order_month'].unique())
if len(train_months_sorted) <= n_val_months + 3:
    raise ValueError("训练月份太少，无法再切验证集")

inner_train_months = train_months_sorted[:-n_val_months]
val_months = train_months_sorted[-n_val_months:]

tr_mask = train['order_month'].isin(inner_train_months)
va_mask = train['order_month'].isin(val_months)

X_tr = X_train.loc[tr_mask].copy()
y_tr = y_train_revenue.loc[tr_mask].copy()
X_va = X_train.loc[va_mask].copy()
y_va = y_train_revenue.loc[va_mask].copy()

# log1p 目标（原始 y 仍保留，用于 MAPE）
y_tr_log = np.log1p(y_tr.astype(float))
y_va_log = np.log1p(y_va.astype(float))
y_train_log = np.log1p(y_train_revenue.astype(float))

# 类别特征（与主流程一致）
cat_features = [c for c in ['customer_hierarchy', 'customer_type', 'quarter', 'month'] if c in X_tr.columns]

def cast_categorical(df, cat_cols, ref_df=None):
    """把类别列转成 category；若给了 ref_df，则用其 categories 对齐"""
    out = df.copy()
    for col in cat_cols:
        if ref_df is None:
            out[col] = out[col].astype("category")
        else:
            out[col] = pd.Categorical(
                out[col],
                categories=ref_df[col].astype("category").cat.categories
            )
    return out

X_tr = cast_categorical(X_tr, cat_features)
X_va = cast_categorical(X_va, cat_features, ref_df=X_tr)

print("inner train:", inner_train_months[0], "~", inner_train_months[-1], "n=", len(X_tr))
print("valid months:", val_months, "n=", len(X_va))
print("cat_features:", cat_features)


def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "learning_rate": trial.suggest_float("learning_rate", 0.03, 0.12, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 5),          # 样本少，限制深度
        "num_leaves": trial.suggest_int("num_leaves", 8, 31),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 20),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 2.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 3.0),
        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
        "min_child_weight": 1e-3,
    }
    
    model = lgb.LGBMRegressor(**params, random_state=42, verbose=-1)
    model.fit(
        X_tr, y_tr_log,
        eval_X=X_va,
        eval_y=y_va_log,
        eval_metric='l1',  # log 域用 l1/l2 更稳；mape 在 log 域含义不同
        categorical_feature=cat_features,
        callbacks=[lgb.early_stopping(80, verbose=False)]
    )
    # pred = model.predict(X_va)
    pred = np.maximum(np.expm1(model.predict(X_va)), 0.0)
    trial.set_user_attr('best_iteration', getattr(model, 'best_iteration_', params['n_estimators']))
    return mean_absolute_percentage_error(y_va, pred)

# ====================== 运行 Optuna ======================
print("开始 Optuna 调参...")
study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=80)

print("Best MAPE:", study.best_value)
print("Best params:", study.best_params)

# ====================== 使用最优参数训练最终模型 ======================
best_params = dict(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    num_leaves=8,
    min_child_samples=15,
    reg_lambda=2.0,
    subsample=0.8,
    colsample_bytree=0.8,
)
# 类别：全量 train + test
X_train_fit = cast_categorical(X_train.copy(), cat_features)
X_test_fit = cast_categorical(X_test.copy(), cat_features, ref_df=X_train_fit)

lgm_model_rev = lgb.LGBMRegressor(
    **best_params,
    random_state=42,
    verbose=-1,
)

# 预测和评估
# 最终训练
lgm_model_rev.fit(X_train_fit, y_train_log)

# ---------- 4) 测试集只评估一次 ----------
lgm_pred_rev_final = np.maximum(np.expm1(lgm_model_rev.predict(X_test_fit)), 0.0)

lgm_mape = mean_absolute_percentage_error(y_test_revenue, lgm_pred_rev_final)
lgm_mae = mean_absolute_error(y_test_revenue, lgm_pred_rev_final)
lgm_bias = (lgm_pred_rev_final.sum() - y_test_revenue.sum()) / y_test_revenue.sum()

print(f"TEST Bias: {lgm_bias:.2%}")
print(f"TEST MAPE: {lgm_mape:.2%}")
print(f"TEST MAE: {lgm_mae:,.2f}")  # 件数，不要写 $
# 偏差校准
lgm_train_pred = np.maximum(np.expm1(lgm_model_rev.predict(X_train_fit)), 0.0)
lgm_train_bias = (lgm_train_pred - y_train_revenue).mean()
print(f"训练集平均偏差: {lgm_train_bias:,.0f}")

lgm_pred_revenue_calibrated = lgm_pred_rev_final - lgm_train_bias

print("\n=== 最终预测 vs 真实值 ===")
print("预测值:", np.round(lgm_pred_revenue_calibrated, 2))
print("真实值:", y_test_revenue.values)
print("预测范围:", lgm_pred_revenue_calibrated.min(), "→", lgm_pred_revenue_calibrated.max())

### catboost

In [ ]:
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

# ==================== cluster 月销售额预测 ====================
cluster_cols = ["customer_hierarchy", "customer_type"]

# 这些列作为类别特征
cat_features = [c for c in ["customer_hierarchy", "customer_type", "quarter", "month"] if c in X_train.columns]

# 复制一份，避免改坏原始数据
X_train_fit = X_train.copy()
X_test_fit = X_test.copy()

# CatBoost 的类别列必须是字符串或整数，不能是 1.0 这种浮点数
for col in cat_features:
    X_train_fit[col] = X_train_fit[col].astype(str)
    X_test_fit[col] = X_test_fit[col].astype(str)

cat_model_rev = CatBoostRegressor(
    iterations=1200,
    learning_rate=0.03,
    depth=4,
    l2_leaf_reg=6,
    min_data_in_leaf=8,
    subsample=0.75,
    random_seed=42,
    verbose=0,
    early_stopping_rounds=80,
    random_strength=1.2,
    bagging_temperature=0.8,
    loss_function="MAE",
    eval_metric="MAPE",
    grow_policy="Lossguide",
    cat_features=cat_features if cat_features else None,
)

# 训练
cat_model_rev.fit(
    X_train_fit,
    y_train_revenue,
    eval_set=(X_test_fit, y_test_revenue),
    use_best_model=True,
    verbose=False,
)

# 预测
cat_pred_rev = np.maximum(cat_model_rev.predict(X_test_fit), 0.0)

# 评估
mape = mean_absolute_percentage_error(y_test_revenue, cat_pred_rev)
mae = mean_absolute_error(y_test_revenue, cat_pred_rev)
bias = (cat_pred_rev.sum() - y_test_revenue.sum()) / y_test_revenue.sum()

print(f"TEST Bias: {bias:.2%}")
print(f"TEST MAPE: {mape:.2%}")
print(f"TEST MAE: {mae:,.2f}")

# 生成每个 cluster 每个月的预测明细
cluster_pred_detail = test[["order_month"] + cluster_cols].copy()
cluster_pred_detail["y_true"] = y_test_revenue.values
cluster_pred_detail["y_pred"] = cat_pred_rev
cluster_pred_detail["abs_error"] = np.abs(cluster_pred_detail["y_true"] - cluster_pred_detail["y_pred"])
cluster_pred_detail["ape"] = cluster_pred_detail["abs_error"] / np.maximum(cluster_pred_detail["y_true"], 1e-6)

print("\n每个 cluster 每月预测明细：")
print(cluster_pred_detail)

# 每个 cluster 汇总表现
cluster_pred_summary = (
    cluster_pred_detail
    .groupby(cluster_cols, as_index=False)
    .agg(
        true_sum=("y_true", "sum"),
        pred_sum=("y_pred", "sum"),
        abs_error_sum=("abs_error", "sum"),
        months=("order_month", "nunique")
    )
)

cluster_pred_summary["bias"] = (
    cluster_pred_summary["pred_sum"] - cluster_pred_summary["true_sum"]
) / cluster_pred_summary["true_sum"]

cluster_pred_summary["wape"] = (
    cluster_pred_summary["abs_error_sum"] / cluster_pred_summary["true_sum"]
)

print("\n每个 cluster 汇总结果：")
print(cluster_pred_summary)

### debug catboost + optuna

In [ ]:
import numpy as np
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
import optuna

# ---------- 1) 训练期内按月切验证（与 LGBM 一致）----------
n_val_months = 3

train_months_sorted = sorted(train['order_month'].unique())
if len(train_months_sorted) <= n_val_months + 3:
    raise ValueError("训练月份太少，无法再切验证集")

inner_train_months = train_months_sorted[:-n_val_months]
val_months = train_months_sorted[-n_val_months:]

tr_mask = train['order_month'].isin(inner_train_months)
va_mask = train['order_month'].isin(val_months)

X_tr = X_train.loc[tr_mask].copy()
y_tr = y_train_revenue.loc[tr_mask].copy()
X_va = X_train.loc[va_mask].copy()
y_va = y_train_revenue.loc[va_mask].copy()

# 类别特征：先定义，再转换（避免用未定义的 X_train_fit）
cat_features = [
    c for c in ['customer_hierarchy', 'customer_type']
    if c in X_tr.columns
]

def cast_categorical(df, cat_cols, ref_df=None):
    out = df.copy()
    for col in cat_cols:
        # 关键：先转成 str，避免 1.0 这种 float 类别
        out[col] = out[col].astype(str)
    if ref_df is None:
        for col in cat_cols:
            out[col] = out[col].astype("category")
    else:
        for col in cat_cols:
            out[col] = pd.Categorical(
                out[col],
                categories=ref_df[col].astype("category").cat.categories
            )
    return out

X_tr = cast_categorical(X_tr, cat_features)
X_va = cast_categorical(X_va, cat_features, ref_df=X_tr)


# ====================== Optuna Objective for CatBoost ======================
def objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 400, 1500),  # 样本不大，不必到 2000
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.12, log=True),
        'depth': trial.suggest_int('depth', 3, 6),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 5, 25),
        'subsample': trial.suggest_float('subsample', 0.7, 0.95),
        'random_strength': trial.suggest_float('random_strength', 0.5, 2.5),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.5),
        'grow_policy': trial.suggest_categorical(
            'grow_policy', ['SymmetricTree', 'Lossguide', 'Depthwise']
        ),
    }
    # Lossguide/Depthwise 有时需要额外参数；先保持简单
    if params['grow_policy'] == 'Lossguide':
        params['max_leaves'] = trial.suggest_int('max_leaves', 16, 64)

    model = CatBoostRegressor(
        **params,
        random_seed=42,
        verbose=0,
        loss_function='MAE',
        eval_metric='MAPE',
        early_stopping_rounds=80,
        # 允许类别
        cat_features=cat_features if cat_features else None,
    )

    model.fit(
        X_tr, y_tr,
        eval_set=(X_va, y_va),   # ✅ 验证集，不是 test
        use_best_model=True,
        verbose=False,
    )
    # pred = model.predict(X_va)
    pred = np.maximum(model.predict(X_va), 0.0)
    return mean_absolute_percentage_error(y_va, pred)


# ====================== 运行 Optuna ======================
print("开始 CatBoost Optuna 调参...")
study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=40, n_jobs=1)      # 可调整为 50~100

print("Best MAPE:", study.best_value)
print("Best params:", study.best_params)


# ====================== 使用最优参数训练最终模型 ======================
best_params = study.best_params.copy()

best_iter = study.best_trial.user_attrs.get("best_iteration")
if best_iter is not None and best_iter > 0:
    best_params["iterations"] = int(best_iter)

X_train_fit = cast_categorical(X_train.copy(), cat_features)
X_test_fit = cast_categorical(X_test.copy(), cat_features, ref_df=X_train_fit)


cat_model_rev = CatBoostRegressor(
    **best_params,
    random_seed=42,
    verbose=0,
    early_stopping_rounds=100,
    loss_function='MAE',
    eval_metric='MAPE',
    cat_features=cat_features if cat_features else None,
)

# 最终训练
cat_model_rev.fit(
    X_train_fit, y_train_revenue,
    verbose=False,
)

# ====================== 预测与评估 ======================
# 原始数据输出，可能包含负数
# cat_pred_rev = cat_model_rev.predict(X_test)
# 数据输出包含非负约束
cat_pred_rev = np.maximum(cat_model_rev.predict(X_test_fit), 0.0)  # ← 不要 expm1


mape = mean_absolute_percentage_error(y_test_revenue, cat_pred_rev)
mae = mean_absolute_error(y_test_revenue, cat_pred_rev)
bias = (cat_pred_rev.sum() - y_test_revenue.sum()) / y_test_revenue.sum()

print(f"TEST Bias: {bias:.2%}")
print(f"TEST MAPE: {mape:.2%}")
print(f"TEST MAE: {mae:,.2f}")  # 件数

# 训练偏差仅作诊断；校准后指标若要用，单独说明是 post-hoc
# cat_train_pred = cat_model_rev.predict(X_train)
cat_train_pred = np.maximum(cat_model_rev.predict(X_train_fit), 0.0)
cat_train_bias = (cat_train_pred - y_train_revenue).mean()
print(f"\n训练集平均偏差: {cat_train_bias:,.2f}")

print("\n预测值:", np.round(cat_pred_rev, 2))
print("真实值:", y_test_revenue.values)

### debug catboost + optuna + log

In [ ]:
import numpy as np
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
import optuna

# ---------- 1) 训练期内按月切验证（与 LGBM 一致）----------
n_val_months = 3

train_months_sorted = sorted(train['order_month'].unique())
if len(train_months_sorted) <= n_val_months + 3:
    raise ValueError("训练月份太少，无法再切验证集")

inner_train_months = train_months_sorted[:-n_val_months]
val_months = train_months_sorted[-n_val_months:]

tr_mask = train['order_month'].isin(inner_train_months)
va_mask = train['order_month'].isin(val_months)

X_tr = X_train.loc[tr_mask].copy()
y_tr = y_train_revenue.loc[tr_mask].copy()
X_va = X_train.loc[va_mask].copy()
y_va = y_train_revenue.loc[va_mask].copy()

# log1p 目标
y_tr_log = np.log1p(y_tr.astype(float))
y_va_log = np.log1p(y_va.astype(float))
y_train_log = np.log1p(y_train_revenue.astype(float))

# CatBoost 类别特征：用列名字符串即可（不必先 astype category）
# cat_features = [c for c in ['customer_hierarchy', 'customer_type'] if c in X_tr.columns]
for col in ['customer_hierarchy', 'customer_type']:
    if col in X_tr.columns:
        X_tr[col] = X_tr[col].astype(str)
        X_va[col] = X_va[col].astype(str)
        X_train_fit[col] = X_train_fit[col].astype(str)
        X_test_fit[col] = X_test_fit[col].astype(str)
# 若 category 是字符串，保持 object/str；product_id 用数值也行，声明为 cat 即可

def cast_categorical(df, cat_cols, ref_df=None):
    out = df.copy()
    for col in cat_cols:
        if ref_df is None:
            out[col] = out[col].astype("category")
        else:
            out[col] = pd.Categorical(
                out[col],
                categories=ref_df[col].astype("category").cat.categories
            )
    return out

X_tr = cast_categorical(X_tr, cat_features)
X_va = cast_categorical(X_va, cat_features, ref_df=X_tr)

# ====================== Optuna Objective for CatBoost ======================
def objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 400, 1500),  # 样本不大，不必到 2000
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.12, log=True),
        'depth': trial.suggest_int('depth', 3, 6),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 5, 25),
        'subsample': trial.suggest_float('subsample', 0.7, 0.95),
        'random_strength': trial.suggest_float('random_strength', 0.5, 2.5),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.5),
        'grow_policy': trial.suggest_categorical(
            'grow_policy', ['SymmetricTree', 'Lossguide', 'Depthwise']
        ),
    }
    # Lossguide/Depthwise 有时需要额外参数；先保持简单
    if params['grow_policy'] == 'Lossguide':
        params['max_leaves'] = trial.suggest_int('max_leaves', 16, 64)

    model = CatBoostRegressor(
        **params,
        random_seed=42,
        verbose=0,
        loss_function='MAE',
        eval_metric='MAPE',
        early_stopping_rounds=80,
        # 允许类别
        cat_features=cat_features if cat_features else None,
    )

    model.fit(
        X_tr, y_tr_log,
        eval_set=(X_va, y_va_log),   # ✅ 验证集，不是 test
        use_best_model=True,
        verbose=False,
    )
    pred = np.maximum(np.expm1(model.predict(X_va)), 0.0)
    trial.set_user_attr("best_iteration", model.get_best_iteration())
    return mean_absolute_percentage_error(y_va, pred)


# ====================== 运行 Optuna ======================
print("开始 CatBoost Optuna 调参...")
study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=40, n_jobs=1)      # 可调整为 50~100

print("Best MAPE:", study.best_value)
print("Best params:", study.best_params)


# ====================== 使用最优参数训练最终模型 ======================
best_params = study.best_params.copy()
best_iter = study.best_trial.user_attrs.get("best_iteration")
if best_iter is not None and best_iter > 0:
    best_params["iterations"] = int(best_iter)

X_train_fit = cast_categorical(X_train.copy(), cat_features)
X_test_fit = cast_categorical(X_test.copy(), cat_features, ref_df=X_train_fit)


cat_model_rev = CatBoostRegressor(
    **best_params,
    random_seed=42,
    verbose=0,
    early_stopping_rounds=100,
    loss_function='MAE',
    eval_metric='MAPE',
    cat_features=cat_features if cat_features else None,
)
for col in cat_features:
    X_train_fit[col] = X_train_fit[col].astype(str)
    X_test_fit[col] = X_test_fit[col].astype(str)
    
# 最终训练
cat_model_rev.fit(
    X_train_fit, y_train_log,
    verbose=False,
)

# ====================== 预测与评估 ======================
cat_pred_rev = np.maximum(np.expm1(cat_model_rev.predict(X_test_fit)), 0.0)


mape = mean_absolute_percentage_error(y_test_revenue, cat_pred_rev)
mae = mean_absolute_error(y_test_revenue, cat_pred_rev)
bias = (cat_pred_rev.sum() - y_test_revenue.sum()) / y_test_revenue.sum()

print(f"TEST Bias: {bias:.2%}")
print(f"TEST MAPE: {mape:.2%}")
print(f"TEST MAE: {mae:,.2f}")  # 件数

# 训练偏差仅作诊断；校准后指标若要用，单独说明是 post-hoc
cat_train_pred = np.maximum(np.expm1(cat_model_rev.predict(X_train_fit)), 0.0)
cat_train_bias = (cat_train_pred - y_train_revenue).mean()
print(f"\n训练集平均偏差: {cat_train_bias:,.2f}")

print("\n预测值:", np.round(cat_pred_rev, 2))
print("真实值:", y_test_revenue.values)

### debug prophet 3 + optuna (best choice)

In [ ]:
import numpy as np
import pandas as pd
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
import optuna
import warnings

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ==========================================================
# Config
# ==========================================================

TARGET = "monthly_revenue"
n_val_months = 3
N_TRIALS = 10

# 如果你要加外生变量，在这里填
USE_REGRESSORS = False
REGRESSORS = []

# ==========================================================
# 以 cluster 为粒度构造数据
# ==========================================================

cluster_cols = ["customer_hierarchy", "customer_type"]

df_cluster_monthly = (
    df_sale_amount_monthly_final[
        ["order_month"] + cluster_cols + [TARGET] + REGRESSORS
    ]
    .copy()
)

df_cluster_monthly = (
    df_cluster_monthly
    .groupby(["order_month"] + cluster_cols, as_index=False)
    .agg({TARGET: "sum", **{c: "sum" for c in REGRESSORS}})
)

# 补全每个 cluster 的月份序列，避免 Prophet 遇到断月
all_months = sorted(df_cluster_monthly["order_month"].unique())
all_clusters = (
    df_cluster_monthly[cluster_cols]
    .drop_duplicates()
    .reset_index(drop=True)
)

full_grid = (
    all_clusters.assign(key=1)
    .merge(pd.DataFrame({"order_month": all_months, "key": 1}), on="key")
    .drop(columns="key")
)

df_cluster_monthly = full_grid.merge(
    df_cluster_monthly,
    on=["order_month"] + cluster_cols,
    how="left"
)

df_cluster_monthly[TARGET] = df_cluster_monthly[TARGET].fillna(0)
for c in REGRESSORS:
    if c in df_cluster_monthly.columns:
        df_cluster_monthly[c] = df_cluster_monthly[c].fillna(0)

# ==========================================================
# 时间划分
# ==========================================================

train_months = sorted(df_cluster_monthly["order_month"].unique())
inner_months = train_months[:-n_val_months]
val_months = train_months[-n_val_months:]

print("Prophet train:", inner_months[0], "~", inner_months[-1])
print("Prophet validation:", val_months)

# ==========================================================
# 转换成 Prophet 格式
# ==========================================================

def to_prophet(df_one):
    g = df_one.sort_values("order_month").copy()
    out = pd.DataFrame({
        "ds": pd.to_datetime(g["order_month"].astype(str) + "-01"),
        "y": g[TARGET].astype(float).values
    })

    for c in REGRESSORS:
        if c not in g.columns:
            raise KeyError(f"Missing regressor {c}")
        out[c] = g[c].astype(float).values

    if out["ds"].duplicated().any():
        raise ValueError("Duplicate month found")
    return out

# ==========================================================
# 缓存每个 cluster 的验证数据
# ==========================================================

tune_cache = {}

for cluster_key, df_cluster in df_cluster_monthly.groupby(cluster_cols):
    train_part = df_cluster[df_cluster["order_month"].isin(inner_months)]
    val_part = df_cluster[df_cluster["order_month"].isin(val_months)]

    if len(train_part) < 8:
        continue

    train_df = to_prophet(train_part)
    future_val = to_prophet(val_part).drop(columns=["y"])

    tune_cache[cluster_key] = (
        train_df,
        future_val,
        val_part[TARGET].values.astype(float)
    )

print("Valid tuning clusters:", len(tune_cache))

# ==========================================================
# Prophet 模型
# ==========================================================

def build_prophet(params):
    model = Prophet(
        growth="linear",
        yearly_seasonality=6,
        weekly_seasonality=False,
        daily_seasonality=False,
        changepoint_prior_scale=params["changepoint_prior_scale"],
        seasonality_prior_scale=params["seasonality_prior_scale"],
        seasonality_mode=params["seasonality_mode"]
    )
    for c in REGRESSORS:
        model.add_regressor(c)
    return model

def prophet_predict(train_df, future_df, params):
    model = build_prophet(params)
    model.fit(train_df)
    forecast = model.predict(future_df)
    return np.maximum(forecast["yhat"].values, 0)

# ==========================================================
# Optuna objective
# ==========================================================

def objective(trial):
    params = {
        "changepoint_prior_scale": trial.suggest_float(
            "changepoint_prior_scale", 0.01, 0.1, log=True
        ),
        "seasonality_prior_scale": trial.suggest_float(
            "seasonality_prior_scale", 0.01, 10, log=True
        ),
        "seasonality_mode": trial.suggest_categorical(
            "seasonality_mode", ["additive"]
        )
    }

    y_true_all = []
    y_pred_all = []

    for cluster_key, (train_df, future_df, y_true) in tune_cache.items():
        try:
            pred = prophet_predict(train_df, future_df, params)
            y_true_all.extend(y_true)
            y_pred_all.extend(pred)
        except Exception:
            continue

    if len(y_true_all) == 0:
        return 999

    return mean_absolute_percentage_error(y_true_all, y_pred_all)

# ==========================================================
# Run Optuna
# ==========================================================

print("\nStart Prophet Optuna...")

study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42)
)

study.optimize(objective, n_trials=N_TRIALS, n_jobs=1)
best_params = study.best_params

print("\nBest validation MAPE:", study.best_value)
print("Best params:", best_params)

# ==========================================================
# 全量 cluster 重新训练 + test 预测
# ==========================================================

result = []

for cluster_key, df_cluster in df_cluster_monthly.groupby(cluster_cols):
    train_cluster = df_cluster[df_cluster["order_month"].isin(train_months[:-n_val_months])]
    test_cluster = df_cluster[df_cluster["order_month"].isin(test_months)].sort_values("order_month")

    if test_cluster.empty:
        continue

    try:
        train_df = to_prophet(train_cluster)
        future_df = to_prophet(test_cluster).drop(columns=["y"])
        pred = prophet_predict(train_df, future_df, best_params)
    except Exception as e:
        print("Failed:", cluster_key, e)
        pred = np.zeros(len(test_cluster))

    tmp = test_cluster[["order_month"] + cluster_cols + [TARGET]].copy()
    tmp["prophet_pred"] = pred
    result.append(tmp)

df_prophet = pd.concat(result, ignore_index=True)

# ==========================================================
# Evaluation
# ==========================================================

eval_df = df_prophet.copy()
mask = eval_df["prophet_pred"].notna()

y_true = eval_df.loc[mask, TARGET].values
y_pred = eval_df.loc[mask, "prophet_pred"].values

mape = mean_absolute_percentage_error(y_true, y_pred)
mae = mean_absolute_error(y_true, y_pred)
bias = (y_pred.sum() - y_true.sum()) / y_true.sum()

print("\n========== Prophet TEST ==========")
print(f"Bias: {bias:.2%}")
print(f"MAPE: {mape:.2%}")
print(f"MAE: {mae:,.2f}")
print("Coverage:", mask.sum(), "/", len(eval_df))

# 最终预测数组，按 cluster + month 排序
prophet_pred_rev = (
    df_prophet
    .sort_values(["order_month"] + cluster_cols)
    ["prophet_pred"]
    .values
)

df_prophet

# print("\n===== 每个 cluster 每月的预测值和真实值 =====")
# print(df_prophet[["order_month"] + cluster_cols + [TARGET, "prophet_pred"]])

print("\n===== 预测值数组 =====")
print(np.round(df_prophet["prophet_pred"].values, 2))

print("\n===== 真实值数组 =====")
print(df_prophet[TARGET].values)

In [ ]:
print("===== 训练集促销特征 =====")
print(train[['promo_paid_qty_ratio', 'promotion_count', 
             'avg_percentage_discount', 'bogo_order_ratio']].describe().round(3))

print("\n===== 测试集促销特征 =====")
print(test[['promo_paid_qty_ratio', 'promotion_count', 
            'avg_percentage_discount', 'bogo_order_ratio']].round(3))

### set final output

In [ ]:
# 方法1：简单加权平均（最常用）
# pred_blend = 0.8 * cat_pred_rev + 0.2 * lgm_pred_rev     # 可以调整权重
# pred_final = 1 * cat_pred_qty + 0 * lgm_pred_qty_final
# pred_final = lgm_pred_rev_final
# pred_final = cat_pred_rev
pred_final = prophet_pred_rev


print("\n融合预测值:", pred_final)

In [ ]:
print("\n1. 特征均值对比:")
for col in ['lag_1_month_revenue', 'rolling_3_month_avg_revenue', 'rolling_6_month_avg_revenue']:
    train_mean = X_train[col].mean()
    test_mean = X_test[col].mean()
    diff_pct = (test_mean - train_mean) / train_mean * 100
    print(f"{col:25s}: 训练={train_mean/1e6:.2f}M, 测试={test_mean/1e6:.2f}M, 差异={diff_pct:+.1f}%")

# 检查目标变量分布
print("\n2. 目标变量对比:")
print(f"训练集平均月销量: {y_train_revenue.mean():,.0f}")
print(f"测试集平均月销量: {y_test_revenue.mean():,.0f}")
print(f"差异: {(y_test_revenue.mean() - y_train_revenue.mean()) / y_train_revenue.mean() * 100:+.1f}%")



### data vis 1

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# 1) 拼成一张评估表（顺序必须与 test 一致）
df_plot = test[['order_month', 'customer_hierarchy', 'customer_type']].copy()
df_plot['y_true'] = np.asarray(y_test_revenue)
df_plot['y_pred'] = np.asarray(pred_final)   # 或 lgm_pred_rev_final / cat_pred_revenue

# 组合 cluster 标识，方便分组和标题
df_plot["cluster"] = (
    df_plot["customer_hierarchy"].astype(str) + " | " + df_plot["customer_type"].astype(str)
)

clusters = sorted(df_plot['cluster'].unique())
n = len(clusters)

# 2) 子图布局：例如 4 列
ncols = 3
nrows = int(np.ceil(n / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(5.0 * ncols, 3.5 * nrows), sharex=False)
axes = np.array(axes).reshape(-1)

for i, cluster in enumerate(clusters):
    ax = axes[i]
    g = (
        df_plot[df_plot['cluster'] == cluster]
        .sort_values('order_month')
    )
    ax.plot(g['order_month'], g['y_true'], marker='o', label='实际值')
    ax.plot(g['order_month'], g['y_pred'], marker='s', linestyle='--', label='预测值')

    title = cluster
    # 单品 MAPE（避免除0）
    mape_i = np.mean(np.abs(g['y_true'] - g['y_pred']) / np.maximum(g['y_true'], 1e-6))
    ax.set_title(f'{title}\nMAPE={mape_i:.1%}', fontsize=10)
    ax.tick_params(axis='x', rotation=45)
    ax.grid(True, alpha=0.3)
    if i == 0:
        ax.legend(fontsize=8)

# 去掉多余空轴
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('每个cluster的月销售金额：预测 vs 实际', fontsize=14, fontweight='bold', y=1.01)
fig.tight_layout()
plt.show()

### data vis 2

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# ---------- 1) 预测 vs 真实（与 test 行顺序一致）----------
df_plot = test[['order_month', 'customer_hierarchy', 'customer_type']].copy()
df_plot['y_true'] = np.asarray(y_test_revenue)
df_plot['y_pred'] = np.asarray(pred_final)  # 或 lgm_pred_revenue_final / cat_pred_revenue

df_plot["cluster"] = (
    df_plot["customer_hierarchy"].astype(str)
    + " | "
    + df_plot["customer_type"].astype(str)
)

# ---------- 2) Baseline：按 customer_hierarchy + customer_type + order_month 对齐 ----------
baseline_cols = [
    'order_month', 'customer_hierarchy', 'customer_type',
    'wma_3', 'ewm_3', 'ewm_6',
]
baseline_cols = [c for c in baseline_cols if c in df_timeseries_features.columns]

df_base = df_timeseries_features[baseline_cols].copy()

df_plot = df_plot.merge(df_base, on=['order_month', 'customer_hierarchy', 'customer_type'], how='left')
df_plot = df_plot.sort_values(['customer_hierarchy', 'customer_type', 'order_month']).reset_index(drop=True)


# 要画的 baseline 列（存在才画）
baseline_plot_cols = [
    c for c in ['wma_3', 'ewm_3', 'ewm_6',
                'rolling_3_month_avg_revenue', 'rolling_6_month_avg_revenue',
                'historical_mean', 'historical_median']
    if c in df_plot.columns
]

# baseline 显示名
baseline_labels = {
    'wma_3': 'WMA(3)',
    'ewm_3': 'EWM(span=3)',
    'ewm_6': 'EWM(span=6)',
    'rolling_3_month_avg_revenue': '3月滚动均值',
    'rolling_6_month_avg_revenue': '6月滚动均值',
    'historical_mean': '历史均值',
    'historical_median': '历史中位数',
}

clusters = sorted(df_plot['cluster'].unique())

# ---------- 3) 每个产品单独一张图 ----------
for cluster in clusters:
    g = df_plot[df_plot['cluster'] == cluster].sort_values('order_month')

    plt.figure(figsize=(10, 5))

    # 真实 & 预测
    plt.plot(g['order_month'], g['y_true'],
             marker='o', linewidth=2.5, color='black', label='实际值', zorder=10)
    plt.plot(g['order_month'], g['y_pred'],
             marker='s', linewidth=2, linestyle='--', label='模型预测', alpha=0.9)

    # baselines
    for col in baseline_plot_cols:
        plt.plot(
            g['order_month'], g[col],
            marker='.', linewidth=1.5, linestyle='-.',
            alpha=0.7, label=baseline_labels.get(col, col)
        )

    # 当前cluster 的 MAPE
    mape_i = np.mean(
        np.abs(g["y_true"] - g["y_pred"]) / np.maximum(g["y_true"].astype(float), 1e-6)
    )
    wape_i = (
        np.abs(g["y_true"] - g["y_pred"]).sum()
        / np.maximum(g["y_true"].astype(float).sum(), 1e-6)
    )

    title_str = f"cluster id={cluster}  |  MAPE={mape_i:.1%}  |  WAPE={wape_i:.1%}"

    plt.title(title_str, fontsize=12, fontweight='bold')
    plt.xlabel('月份')
    plt.ylabel('销售额 (美元)')
    plt.xticks(rotation=45)
    plt.grid(True, alpha=0.3)
    plt.legend(loc='best', fontsize=9, ncol=2)
    plt.tight_layout()
    plt.show()

In [ ]:
# 关闭数据库连接
engine.dispose()